## MLP & RealMLP: Cross-validation & testing
### Preparing the datasets

In [1]:
from src.data_processor import get_prepared_data, _build_preprocessor, DATASET_CONFIG
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, KFold
from sklearn.neural_network import MLPClassifier, MLPRegressor
from scipy.stats import loguniform
import numpy as np
import pandas as pd
from pprint import pprint

#### Classification tasks
* For tabular classification and regression tasks, the following hyperparameters would be of interest during the tuning process:
    + `hidden_layer_sizes`: For tabular data, the width of the hidden layers often matters more than the depth. This parameter directly affects the capacity of the model.
    + `alpha`: This parameter controls the strength of the regularization term in the model.
    + `learning_rate_init`: This parameter is only used when the `Adam` or `SGD` solver is used. The cross validation function tests a number of step sizes.
    + `activation`: Controls the activation layer used in the model: Rectified Linear Unit usually works well, but the `tanh` activation layer can help on small tabular problems.

In [2]:
# Get datasets for classification tasks
datasets_to_run = {
    k: v
    for k, v in DATASET_CONFIG.items()
    if v['task'] == 'classification'
}

# Hyperparameters to test during cross-validation
param_dist = {
    "mlp__hidden_layer_sizes": [(32,), (64,), (64,32), (128,64)],
    "mlp__alpha": loguniform(1e-5, 1e-1),
    "mlp__learning_rate_init": loguniform(1e-4, 1e-2),
    "mlp__activation": ["relu", "tanh"],
}

for dataset_name, config in datasets_to_run.items():
    print(f'Loading and processing: {dataset_name}...')
    try:
        X_train, X_val, X_test, y_train, y_val, y_test = get_prepared_data(dataset_name, return_raw=True)
    except Exception as e:
        print(f'Skipping {dataset_name} due to error: {e}')
        continue

    X_trainval = pd.concat([X_train, X_val], axis=0)
    y_trainval = np.concatenate([y_train, y_val], axis=0)

    pipe = Pipeline([
        ('preprocess', _build_preprocessor(X_trainval)),
        ('mlp', MLPClassifier(max_iter=2000, early_stopping=True, random_state=42)),
    ])

    search = RandomizedSearchCV(
        pipe, param_dist, n_iter=60,
        cv=StratifiedKFold(5, shuffle=True, random_state=42),
        scoring='roc_auc', n_jobs=-1, random_state=42
    )
    print(f'Fitting {dataset_name}...')
    search.fit(X_trainval, y_trainval)

    print(f'Best parameters: ')
    pprint(search.best_params_)
    print(f'Best AUC-ROC: ')
    print(search.best_score_)
    print('')

Loading and processing: Classification_n_gt_10k...
Fitting Classification_n_gt_10k...
Best parameters: 
{'mlp__activation': 'tanh',
 'mlp__alpha': np.float64(0.0017910276390244793),
 'mlp__hidden_layer_sizes': (128, 64),
 'mlp__learning_rate_init': np.float64(0.006533305220227739)}
Best AUC-ROC: 
0.9829617463591764

Loading and processing: Classification_d_gt_50...
Fitting Classification_d_gt_50...
Best parameters: 
{'mlp__activation': 'tanh',
 'mlp__alpha': np.float64(0.033256102959751885),
 'mlp__hidden_layer_sizes': (64,),
 'mlp__learning_rate_init': np.float64(0.004132765459466363)}
Best AUC-ROC: 
0.9292507360660242

Loading and processing: Classification_Mixed...
Fitting Classification_Mixed...
Best parameters: 
{'mlp__activation': 'relu',
 'mlp__alpha': np.float64(2.6422690597255385e-05),
 'mlp__hidden_layer_sizes': (128, 64),
 'mlp__learning_rate_init': np.float64(0.004048966222584676)}
Best AUC-ROC: 
0.9316116526454381



#### Regression tasks

In [3]:
# Get datasets for regression tasks
datasets_to_run_reg = {
    k: v
    for k, v in DATASET_CONFIG.items()
    if v['task'] == 'regression'
}

for dataset_name, config in datasets_to_run_reg.items():
    print(f'Loading and processing: {dataset_name}...')
    try:
        X_train, X_val, X_test, y_train, y_val, y_test = get_prepared_data(dataset_name, return_raw=True)
    except Exception as e:
        print(f'Skipping {dataset_name} due to error: {e}')
        continue

    X_trainval = pd.concat([X_train, X_val], axis=0)
    y_trainval = np.concatenate([y_train, y_val], axis=0)

    pipe = Pipeline([
        ('preprocess', _build_preprocessor(X_trainval)),
        ('mlp', MLPRegressor(max_iter=2000, early_stopping=True, random_state=42)),
    ])

    search = RandomizedSearchCV(
        pipe, param_dist, n_iter=60,
        cv=KFold(5, shuffle=True, random_state=42),
        scoring='neg_root_mean_squared_error', n_jobs=-1, random_state=42
    )
    print(f'Fitting {dataset_name}...')
    search.fit(X_trainval, y_trainval)

    print(f'Best parameters: ')
    pprint(search.best_params_)
    print(f'Best RMSE: ')
    print(-search.best_score_)
    print('')

Loading and processing: Regression_n_gt_10k...
Fitting Regression_n_gt_10k...
Best parameters: 
{'mlp__activation': 'relu',
 'mlp__alpha': np.float64(0.00011957746304562952),
 'mlp__hidden_layer_sizes': (128, 64),
 'mlp__learning_rate_init': np.float64(0.007688106801474953)}
Best RMSE: 
4.016477791541748

Loading and processing: Regression_d_gt_50...
Fitting Regression_d_gt_50...


/Users/bachvu7723/Downloads/At the moment/SCHOOL DIRECTORY/2026/T1 2026/COMP9417/WORKING_DIRECTORY/assignments/COMP9417-Group-Project/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/bachvu7723/Downloads/At the moment/SCHOOL DIRECTORY/2026/T1 2026/COMP9417/WORKING_DIRECTORY/assignments/COMP9417-Group-Project/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/bachvu7723/Downloads/At the moment/SCHOOL DIRECTORY/2026/T1 2026/COMP9417/WORKING_DIRECTORY/assignments/COMP9417-Group-Project/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iteration

Best parameters: 
{'mlp__activation': 'relu',
 'mlp__alpha': np.float64(0.00011957746304562952),
 'mlp__hidden_layer_sizes': (128, 64),
 'mlp__learning_rate_init': np.float64(0.007688106801474953)}
Best RMSE: 
31053.89271809566

Loading and processing: Regression_Mixed...
Fitting Regression_Mixed...


/Users/bachvu7723/Downloads/At the moment/SCHOOL DIRECTORY/2026/T1 2026/COMP9417/WORKING_DIRECTORY/assignments/COMP9417-Group-Project/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/bachvu7723/Downloads/At the moment/SCHOOL DIRECTORY/2026/T1 2026/COMP9417/WORKING_DIRECTORY/assignments/COMP9417-Group-Project/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/bachvu7723/Downloads/At the moment/SCHOOL DIRECTORY/2026/T1 2026/COMP9417/WORKING_DIRECTORY/assignments/COMP9417-Group-Project/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iteration

Best parameters: 
{'mlp__activation': 'relu',
 'mlp__alpha': np.float64(0.00011957746304562952),
 'mlp__hidden_layer_sizes': (128, 64),
 'mlp__learning_rate_init': np.float64(0.007688106801474953)}
Best RMSE: 
5095.028532223107

